# Demo 07 : Buyer and seller

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


In [2]:
import json
import operator
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for batch in response.generations:
                for generation in batch:
                    usage = generation.message.usage_metadata or {}
                    self.input_tokens += usage.get("input_tokens", 0)
                    self.output_tokens += usage.get("output_tokens", 0)


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=500, max_retries=0)
started = time.perf_counter()
DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))
NOTEBOOK_DIR = ROOT / "notebooks" / "06-agent-communication" if ROOT else Path.cwd()


def stock(part="CE-CF-02"):
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        db.row_factory = sqlite3.Row
        row = db.execute("SELECT part_no, name, stock, price_eur, depot FROM parts WHERE part_no = ?", (part,)).fetchone()
    return dict(row) if row else {"part_no": part, "error": "unknown part"}


def guidance(error="F-CE-110"):
    return "\n".join(f"[{p.name}] {line}" for p in sorted(MANUALS.glob("*.md"))
                     for line in p.read_text(encoding="utf-8").splitlines()
                     if line.startswith("| " + error + " |"))


FACTS = stock()
QUESTION = "Report CE-CF-02 stock and location, then the first action and replacement condition for F-CE-110."
LIMIT = {"recursion_limit": 16}


def finish_metrics(name):
    row = {"model": MODEL, "rps": float(os.environ.get("COURSE_RPS", "2")),
           "calls": counter.calls, "input_tokens": counter.input_tokens,
           "output_tokens": counter.output_tokens, "seconds": round(time.perf_counter() - started, 1)}
    print("metrics :", row)
    (NOTEBOOK_DIR / (name + "-metrics.json")).write_text(json.dumps(row, indent=2) + "\n", encoding="utf-8")

## a · the-conflict

We adapt the seller and buyer from Yair's crew to Veenhaven Bus. Both serve one owner with competing objectives. We use a made-up reserve and record proposals without changing the service database.

In [3]:
from pydantic import BaseModel, Field


class Proposal(BaseModel):
    quantity: int = Field(ge=0, le=6)
    reason: str


class FleetState(TypedDict):
    facts: dict
    reserve: int
    proposals: Annotated[list[dict], operator.add]
    settlement: dict


def propose(role):
    goal = {"seller": "Propose a quantity to sell to reduce stored surplus.",
            "buyer": "Propose a quantity to keep for workshop continuity."}[role]
    def node(state):
        proposal = llm.with_structured_output(Proposal).invoke(goal + " Give a reason. This is a proposal only.\n"
            + json.dumps({"facts": state["facts"], "reserve": state["reserve"]}))
        return {"proposals": [{"role": role, **proposal.model_dump()}]}
    return node


def settle(state):
    proposals = {p["role"]: p for p in state["proposals"]}
    available = state["facts"]["stock"]
    retained = max(state["reserve"], min(available, proposals["buyer"]["quantity"]))
    sale = min(proposals["seller"]["quantity"], max(0, available - retained))
    return {"settlement": {"sell": sale, "retain": available - sale, "external_action": False}}


fleet = StateGraph(FleetState)
for role in ["seller", "buyer"]:
    fleet.add_node(role, propose(role))
    fleet.add_edge(START, role)
fleet.add_node("settle", settle)
fleet.add_edge(["seller", "buyer"], "settle")
fleet.add_edge("settle", END)
result = fleet.compile().invoke({"facts": FACTS, "reserve": 4, "proposals": []})
print("facts :", FACTS)
for proposal in sorted(result["proposals"], key=lambda p: p["role"]):
    print("proposal :", proposal)
print("settlement :", result["settlement"])

facts : {'part_no': 'CE-CF-02', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}
proposal : {'role': 'buyer', 'quantity': 4, 'reason': 'Keep the stated reserve of 4 traction cooling fans for workshop continuity; 6 are currently in stock.'}
proposal : {'role': 'seller', 'quantity': 2, 'reason': 'Stock is 6 and the reserve is 4, so selling 2 units reduces the surplus while preserving the reserve.'}
settlement : {'sell': 2, 'retain': 4, 'external_action': False}


## b · least-privilege-per-agent

We expose separate tools and use trusted runtime context to select a role's credential scope. The credential labels below are synthetic. An enforcement function checks the scope even if a caller requests the other role's operation.

In [4]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool


@dataclass(frozen=True)
class Principal:
    role: str
    credential_scope: str


def authorise(principal, operation):
    allowed = {"seller": ("sales-draft-only", "draft-sale"),
               "buyer": ("procurement-draft-only", "draft-purchase")}
    if allowed.get(principal.role) != (principal.credential_scope, operation):
        raise PermissionError("Operation outside this principal's scope.")


@tool
def draft_sale(quantity: int, runtime: ToolRuntime[Principal]) -> str:
    """Draft a CE-CF-02 sale proposal. No external action."""
    authorise(runtime.context, "draft-sale")
    return json.dumps({"principal": runtime.context.role, "operation": "draft-sale", "quantity": quantity,
                       "part_no": "CE-CF-02", "external_action": False})


@tool
def draft_purchase(quantity: int, runtime: ToolRuntime[Principal]) -> str:
    """Draft a CE-DA-03 purchase proposal. No external action."""
    authorise(runtime.context, "draft-purchase")
    return json.dumps({"principal": runtime.context.role, "operation": "draft-purchase", "quantity": quantity,
                       "part_no": "CE-DA-03", "external_action": False})


seller = create_agent(llm, tools=[draft_sale], context_schema=Principal,
    system_prompt="Draft the requested sale with draft_sale. Do not claim a sale completed.")
buyer = create_agent(llm, tools=[draft_purchase], context_schema=Principal,
    system_prompt="Draft the requested purchase with draft_purchase. Do not claim an order completed.")
for agent, principal, task in [
    (seller, Principal("seller", "sales-draft-only"), "Draft a sale of two CE-CF-02 cooling fans."),
    (buyer, Principal("buyer", "procurement-draft-only"), "Draft a purchase of one CE-DA-03 door actuator."),
]:
    output = agent.invoke({"messages": [HumanMessage(task)]}, LIMIT, context=principal)
    print("principal :", principal)
    print("tool results :", [(m.name, m.content) for m in output["messages"] if isinstance(m, ToolMessage)])
    print("answer :", output["messages"][-1].content)
print("seller schema :", draft_sale.args)
print("buyer schema :", draft_purchase.args)
try:
    authorise(Principal("seller", "sales-draft-only"), "draft-purchase")
except PermissionError as exc:
    print("denied :", str(exc))

principal : Principal(role='seller', credential_scope='sales-draft-only')
tool results : [('draft_sale', '{"principal": "seller", "operation": "draft-sale", "quantity": 2, "part_no": "CE-CF-02", "external_action": false}')]
answer : Drafted a sale proposal for two CE-CF-02 cooling fans. No sale was completed.


principal : Principal(role='buyer', credential_scope='procurement-draft-only')
tool results : [('draft_purchase', '{"principal": "buyer", "operation": "draft-purchase", "quantity": 1, "part_no": "CE-DA-03", "external_action": false}')]
answer : Draft purchase proposal created for 1 CE-DA-03 door actuator. No order was placed.
seller schema : {'quantity': {'title': 'Quantity', 'type': 'integer'}}
buyer schema : {'quantity': {'title': 'Quantity', 'type': 'integer'}}
denied : Operation outside this principal's scope.


We simulate role-scoped credentials and enforce operations in code. Production isolation also needs separate service identities or browser profiles and downstream authorisation. A prompt, hidden tool schema or approval gate alone cannot contain every shared credential.

## Things to try

Optional : we can change the part, the context boundary or the policy and inspect the result.

In [5]:
finish_metrics("demo07")

metrics : {'model': 'openai/gpt-6-luna', 'rps': 0.4, 'calls': 6, 'input_tokens': 702, 'output_tokens': 161, 'seconds': 16.3}
